# Data quality

## 1. Load files

In [1]:
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import col

spark = pyspark.sql.SparkSession.builder.appName("dev").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "../data"
attributes = spark.read.csv(f"{DATA_DIR}/features_attributes.csv", header=True, inferSchema=True)
financials = spark.read.csv(f"{DATA_DIR}/features_financials.csv", header=True, inferSchema=True)
print(attributes.count(), financials.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/25 15:01:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


12500 12500


In [2]:
attributes.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)



In [4]:
financials.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: string (nullable = true)
 |-- Monthly_Inhand_Salary: double (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: string (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: string (nullable = true)
 |-- Changed_Credit_Limit: string (nullable = true)
 |-- Num_Credit_Inquiries: double (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: string (nullable = true)
 |-- Credit_Utilization_Ratio: double (nullable = true)
 |-- Credit_History_Age: string (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month: double (nullable = true)
 |-- Amount_invested_monthly: string (nullable = true)
 |-- Payment_Behaviour: string (nullable = true)
 |-- Mont

## 2. Numbers stored as text

In [6]:
def non_numeric(df, column):
    """Identify non-numeric values within a PySpark DataFrame column"""
    bad = df.filter(col(column).cast("double").isNull())
    print(f"{column}: {bad.count()} values are not numbers")
    bad.groupBy(F.regexp_replace(column, "[0-9]", "9").alias("pattern")).count().orderBy(F.desc("count")).show(5, truncate=False)

non_numeric(attributes, "Age")

for c in ["Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment", "Changed_Credit_Limit", "Outstanding_Debt", "Amount_invested_monthly", "Monthly_Balance"]:
    non_numeric(financials, c)


Age: 637 values are not numbers
+-------+-----+
|pattern|count|
+-------+-----+
|99_    |624  |
|9999_  |13   |
+-------+-----+

Annual_Income: 859 values are not numbers
+----------+-----+
|pattern   |count|
+----------+-----+
|99999.99_ |565  |
|99999.9_  |87   |
|999999.99_|79   |
|99999.999_|34   |
|9999.99_  |27   |
+----------+-----+
only showing top 5 rows

Num_of_Loan: 623 values are not numbers
+-------+-----+
|pattern|count|
+-------+-----+
|9_     |619  |
|999_   |2    |
|99_    |1    |
|9999_  |1    |
+-------+-----+

Num_of_Delayed_Payment: 374 values are not numbers
+-------+-----+
|pattern|count|
+-------+-----+
|99_    |261  |
|9_     |105  |
|-9_    |4    |
|999_   |2    |
|9999_  |2    |
+-------+-----+

Changed_Credit_Limit: 254 values are not numbers
+-------+-----+
|pattern|count|
+-------+-----+
|_      |254  |
+-------+-----+

Outstanding_Debt: 139 values are not numbers
+--------+-----+
|pattern |count|
+--------+-----+
|9999.99_|70   |
|999.99_ |51   |
|99.99_ 

**Findings:** 
- Most bad values are numbers with trailing `_` (e.g. `23_`, `52312.68_`)
- 3 are not numbers: `Changed_Credit_Limit` = `_` (254 rows), `Amount_invested_monthly` = `__10000__` (558 rows) and
`Monthly_Balance` = `__-333333333333333333333333333__` (1 row)
- Removing the underscores would turn the last 2 into numbers that look real, so they need their own rule

## 3. Ranges

In [10]:
text_numbers = ["Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment", "Changed_Credit_Limit", "Outstanding_Debt", "Amount_invested_monthly", "Monthly_Balance"]

numeric_financials = financials
for c in text_numbers:
    numeric_financials = numeric_financials.withColumn(c, F.regexp_replace(col(c), "_", "").cast("double"))

numeric_columns = [c for c, t in numeric_financials.dtypes if t in ("int", "double")]
numeric_financials.select(numeric_columns).summary("count", "min", "1%", "25%", "50%", "75%", "99%", "max").toPandas().set_index("summary").T


summary,count,min,1%,25%,50%,75%,99%,max
Annual_Income,12500,7005.93,7536.015,19453.05,37551.78,72674.76,178793.92,2.3834698E7
Monthly_Inhand_Salary,12500,303.6454166666666,531.0429166666668,1624.7516666666668,3085.853333333333,5944.243333333333,13785.103333333333,15204.633333333331
Num_Bank_Accounts,12500,-1,0,3,6,7,489,1756
Num_Credit_Card,12500,0,1,4,5,7,874,1499
Interest_Rate,12500,1,1,8,14,20,2851,5789
Num_of_Loan,12500,-100.0,-100.0,1.0,3.0,5.0,9.0,1495.0
Delay_from_due_date,12500,-5,0,10,18,28,61,67
Num_of_Delayed_Payment,12500,-3.0,0.0,9.0,14.0,18.0,27.0,4293.0
Changed_Credit_Limit,12246,-6.49,-1.86,5.369999999999999,9.41,14.94,28.79,36.97
Num_Credit_Inquiries,12500,0.0,0.0,4.0,6.0,10.0,884.0,2554.0


In [11]:
ages = attributes.withColumn("Age", F.regexp_replace("Age", "_", "").cast("int"))
ages.select("Age").summary("min", "1%", "5%", "50%", "95%", "99%", "max").show()

+-------+----+
|summary| Age|
+-------+----+
|    min|-500|
|     1%|  14|
|     5%|  17|
|    50%|  33|
|    95%|  54|
|    99%|3899|
|    max|8678|
+-------+----+



In [13]:
ages.filter((col("Age") < 18) | (col("Age") > 100)).groupBy("Age").count().orderBy("Age").show(8)
print("Age above 100:", ages.filter(col("Age") > 100).count())

+----+-----+
| Age|count|
+----+-----+
|-500|  104|
|  14|   87|
|  15|  195|
|  16|  189|
|  17|  198|
| 169|    1|
| 203|    1|
| 210|    1|
+----+-----+
only showing top 8 rows

Age above 100: 215


In [14]:
numeric_financials.groupBy("Num_Bank_Accounts").count().orderBy("Num_Bank_Accounts").show(16)
print("more than 10 bank accounts:", numeric_financials.filter(col("Num_Bank_Accounts") > 10).count())

+-----------------+-----+
|Num_Bank_Accounts|count|
+-----------------+-----+
|               -1|    4|
|                0|  542|
|                1|  566|
|                2|  543|
|                3| 1482|
|                4| 1529|
|                5| 1518|
|                6| 1621|
|                7| 1594|
|                8| 1593|
|                9|  687|
|               10|  651|
|               11|    3|
|               26|    1|
|               28|    1|
|               50|    1|
+-----------------+-----+
only showing top 16 rows

more than 10 bank accounts: 170


In [15]:
numeric_financials.groupBy("Num_of_Loan").count().orderBy("Num_of_Loan").show(12)

+-----------+-----+
|Num_of_Loan|count|
+-----------+-----+
|     -100.0|  500|
|        0.0| 1363|
|        1.0| 1324|
|        2.0| 1879|
|        3.0| 1887|
|        4.0| 1844|
|        5.0|  894|
|        6.0|  976|
|        7.0|  916|
|        8.0|  395|
|        9.0|  455|
|       18.0|    1|
+-----------+-----+
only showing top 12 rows



In [16]:
income_check = numeric_financials.withColumn(
    "ratio", col("Annual_Income") / (col("Monthly_Inhand_Salary") * 12))
income_check.select("ratio").summary("min", "1%", "50%", "99%", "max").show()
print("annual income over twice 12 x monthly salary:", income_check.filter(col("ratio") > 2).count())

+-------+------------------+
|summary|             ratio|
+-------+------------------+
|    min|0.6741517186429752|
|     1%|0.7681126128345944|
|    50%|1.0034731713145224|
|    99%|1.6932165379149473|
|    max| 3859.589455797814|
+-------+------------------+

annual income over twice 12 x monthly salary: 114


**Findings:** 
- Most numeric columns have a plausible body and a thin tail of impossible values: 75% of customers have at most 7 bank accounts, yet 170 have more than 10 and the maximum is 1,756.
- `Age` has 104 rows of -500 and 215 above 100 (up to 8,678); ages 14-17 occur 87-198 times each, so they look like part of the data
- `Num_of_Loan` has 500 rows of -100, which might be a sentinel value
- Negative `Delay_from_due_date` (paid early) and `Changed_Credit_Limit`(limit reduced) are plausible; negative counts are not
- `Annual_Income` is more than twice 12 months' `Monthly_Inhand_Salary` for 114 customers.

## 4. Categories and text columns

In [17]:
attributes.groupBy("Occupation").count().orderBy(F.desc("count")).show(truncate=False)

+-------------+-----+
|Occupation   |count|
+-------------+-----+
|_______      |880  |
|Lawyer       |828  |
|Architect    |795  |
|Engineer     |793  |
|Accountant   |791  |
|Scientist    |789  |
|Teacher      |782  |
|Media_Manager|780  |
|Developer    |780  |
|Mechanic     |780  |
|Entrepreneur |776  |
|Journalist   |761  |
|Doctor       |760  |
|Musician     |741  |
|Manager      |736  |
|Writer       |728  |
+-------------+-----+



In [18]:
for c in ["Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]:
    financials.groupBy(c).count().orderBy(F.desc("count")).show(truncate=False)

+----------+-----+
|Credit_Mix|count|
+----------+-----+
|Standard  |4497 |
|Good      |3032 |
|_         |2611 |
|Bad       |2360 |
+----------+-----+

+---------------------+-----+
|Payment_of_Min_Amount|count|
+---------------------+-----+
|Yes                  |6571 |
|No                   |4491 |
|NM                   |1438 |
+---------------------+-----+

+--------------------------------+-----+
|Payment_Behaviour               |count|
+--------------------------------+-----+
|Low_spent_Small_value_payments  |3202 |
|High_spent_Medium_value_payments|2242 |
|Low_spent_Medium_value_payments |1686 |
|High_spent_Large_value_payments |1683 |
|High_spent_Small_value_payments |1389 |
|Low_spent_Large_value_payments  |1300 |
|!@9#%8                          |998  |
+--------------------------------+-----+



In [20]:
financials.select("Credit_History_Age", "Type_of_Loan").show(5, truncate=False)
print("Credit_History_Age not in 'N Years and M Months' form:",
      financials.filter(~col("Credit_History_Age").rlike(r"^\d+ Years and \d+ Months$")).count())

+----------------------+---------------------------------------------------------------------+
|Credit_History_Age    |Type_of_Loan                                                         |
+----------------------+---------------------------------------------------------------------+
|10 Years and 9 Months |Credit-Builder Loan, and Home Equity Loan                            |
|31 Years and 0 Months |Not Specified, Home Equity Loan, Credit-Builder Loan, and Payday Loan|
|15 Years and 10 Months|NULL                                                                 |
|15 Years and 10 Months|Student Loan, Credit-Builder Loan, and Debt Consolidation Loan       |
|17 Years and 10 Months|Student Loan, Debt Consolidation Loan, and Personal Loan             |
+----------------------+---------------------------------------------------------------------+
only showing top 5 rows

Credit_History_Age not in 'N Years and M Months' form: 0


In [21]:
numeric_financials.filter(col("Type_of_Loan").isNull()).groupBy("Num_of_Loan").count().orderBy(F.desc("count")).show(5)

+-----------+-----+
|Num_of_Loan|count|
+-----------+-----+
|        0.0| 1363|
|     -100.0|   53|
|      439.0|    1|
|     1074.0|    1|
|       18.0|    1|
+-----------+-----+
only showing top 5 rows



**Findings:**
- Placeholders in categorical columns: `Occupation` = `_______`(880), `Credit_Mix` = `_` (2611) and `Payment_Behaviour` = `!@9#%8` (998)
- `Payment_of_Min_Amount` has `NM` (1438) besides Yes/No
- `Credit_History_Age` is consistently "# Years and # Months"
- `Type_of_Loan` is a comma-separated list; 1363 of 1426 nulls belong to customers with `Num_of_Loan` = 0, so a null there mostly means "no loans".

## 5. Identifiers

In [23]:
attributes.filter(~col("SSN").rlike(r"^\d{3}-\d{2}-\d{4}$")).groupBy("SSN").count().show()

+---------+-----+
|      SSN|count|
+---------+-----+
|#F%$D@*&8|  703|
+---------+-----+



In [31]:
odd_names = attributes.filter(~col("Name").rlike(r"^[A-Za-z .'-]+$"))
print("names with other characters:", odd_names.count())
print("names containing a double quote:", attributes.filter(col("Name").contains('"')).count())
odd_names.select("Name").show(8)

no_quote_names = odd_names.filter(~col("Name").contains('"'))
no_quote_names.select("Name").show()

names with other characters: 119
names containing a double quote: 107
+--------------------+
|                Name|
+--------------------+
|       "ODonnell""d"|
|         "OLeary""k"|
|       "ODonnell""i"|
|           """Johnt"|
|         """Silviaz"|
|           """Johns"|
|"""John ODonnell""q"|
|"""Eileen OGrady""v"|
+--------------------+
only showing top 8 rows

+-----------------+
|             Name|
+-----------------+
|          , Asiaf|
|          , Asiaw|
|Jeremy Wagstaff,n|
|        , Globalf|
|      , Europeanx|
|          , Asiac|
|        , Globalw|
|        , Globalf|
|        , Globalt|
|        , Globalq|
|        , Globali|
|              , e|
+-----------------+



In [27]:
attributes_escaped = spark.read.option("escape", '"').csv(f"{DATA_DIR}/features_attributes.csv", header=True, inferSchema=True)
attributes_escaped.filter(col("Name").contains('"')).select("Customer_ID", "Name").show(4)

+-----------+----------+
|Customer_ID|      Name|
+-----------+----------+
| CUS_0x138d|ODonnell"d|
| CUS_0x162f|  OLeary"k|
| CUS_0x1680|ODonnell"i|
| CUS_0x1890|    "Johnt|
+-----------+----------+
only showing top 4 rows



**Findings:** 
- Every SSN follows the `123-45-6789` pattern except `#F%$D@*&8` (703 customers)
- 119 names contain special characters: 107 have a double quotes "" and 12 are fragments (10 of which seem like regions such as `, Asiaf`, `Globalf`)
- Spark's CSV reader shows the quoted names with CSV quoting (`"ODonnell""d"`); use `option("escape", '"')` to read it as `ODonnell"d` like pandas (https://towardsdev.com/read-csv-files-using-pyspark-handling-quoted-and-delimited-values-75d47985a5e7)

## 6. Does the placeholders affect the same rows?

In [28]:
quality = attributes.join(financials, on="Customer_ID").select(
    "Customer_ID",
    (col("SSN") == "#F%$D@*&8").cast("int").alias("ssn"),
    (col("Occupation") == "_______").cast("int").alias("occupation"),
    (col("Credit_Mix") == "_").cast("int").alias("credit_mix"),
    (col("Payment_Behaviour") == "!@9#%8").cast("int").alias("payment_behaviour"),
    (col("Amount_invested_monthly") == "__10000__").cast("int").alias("amount_invested"),
    (col("Changed_Credit_Limit") == "_").cast("int").alias("credit_limit"),
)
placeholders = quality.columns[1:]
print("customers:", quality.count())

# See distribution of dirty data in valid and invalid SSN rows
quality.groupBy("ssn").agg(*[F.round(F.avg(c), 3).alias(c) for c in placeholders[1:]]).show()

customers: 12500
+---+----------+----------+-----------------+---------------+------------+
|ssn|occupation|credit_mix|payment_behaviour|amount_invested|credit_limit|
+---+----------+----------+-----------------+---------------+------------+
|  1|     0.074|     0.216|            0.077|          0.044|        0.01|
|  0|      0.07|     0.208|             0.08|          0.045|       0.021|
+---+----------+----------+-----------------+---------------+------------+



In [33]:
# See number of placeholders each customer has
quality = quality.withColumn("placeholders", sum(col(c) for c in placeholders))
quality.groupBy("placeholders").count().orderBy("placeholders").show()
print("Number of customers with at least 1 placeholder: ", quality.filter(col("placeholders") >= 1).count())

+------------+-----+
|placeholders|count|
+------------+-----+
|           0| 7432|
|           1| 4226|
|           2|  754|
|           3|   82|
|           4|    6|
+------------+-----+

Number of customers with at least 1 placeholder:  5068


**Findings:**
- Placeholders are scattered independently
- Dropping all placeholder rows will discard 41% of customers (5068)